### How LLMs Behave



**1. Tokens**
- Core Unit: Tokens are the fundamental unit of input, output, pricing, and context limits in LLMs—not raw characters or words.

- Variable Length: The average characters per token varies depending on the specific model's tokenizer.

- Scope: Everything processed by Claude—including prompts, conversation history, tool definitions, tool results, and generated responses—is measured and billed in tokens.

**2. Context Window**
- Input Limit: Represents the maximum capacity of tokens an LLM can retain and process within a single prompt or active conversation session.

- Budgeting: Managing token counts within this limit is essential for feature design, avoiding context truncation, and keeping API costs predictable.

**3. Sampling**
- Generation Process: The probabilistic method by which an LLM selects the next token from predicted probability distributions.

- Parameters: Hyperparameters like temperature and top_p control output variability, balancing predictability/coherence with creativity.

**4. Non-determinism**
- Variability: Because LLMs generate responses using probability-based sampling, identical inputs can produce slightly different outputs across runs.

- Engineering Considerations: Mitigating non-determinism requires careful prompt structure, setting temperature to zero where consistency is needed, and relying on structured outputs (e.g., tool calls or JSON schema).

### Models And Reasoning

**1. The Claude Model Family**

The model family spans four main tiers, balancing cost, latency, and capability:

* **Fable:** The most capable tier, built for the most demanding reasoning, coding, and agentic tasks where maximum intelligence is required.
* **Opus:** Designed for demanding, high-complexity workloads above the Sonnet capability envelope.
* **Sonnet:** The balanced default for the majority of production workloads.
* **Haiku:** Built for high speed and cost efficiency on straightforward tasks that fit its capability envelope.

**2. Practical Selection Strategy**

* **Default Starting Point:** Always start development and evaluation with **Sonnet**.
* **Scaling Up:** Move up a tier (to Opus or Fable) only when evaluation metrics show that Sonnet falls short of your quality threshold.
* **Scaling Down:** Move down a tier (to Haiku) only when evaluations demonstrate that any drop in output quality is acceptable for your specific task requirements.

**Summary: Reasoning Modes & How They Work Together**

This section covers how **Reasoning Modes** function and interact with **Model Selection** within **Module 1: MSO Foundations**:

**1. Reasoning Modes (Adaptive Thinking)**

* **Independent Axis:** Reasoning is configured **per request** and is entirely separate from model selection.
* **Adaptive Thinking:** Modern Claude models dynamically decide *when* and *how much* to think based on the task.
* **Depth Control via Effort:** Reasoning depth is tuned using an `effort` setting (e.g., `low`, `medium`, `high`, or `max`) rather than fixed token budgets. The older `budget_tokens` parameter is deprecated and triggers a `400` error on newer model generations.
* **Display Handling:** Internal thinking blocks/monologues are omitted from responses by default on newer models; you must explicitly request summarized display if you need to render it in your UI.
* **Cost Allocation:** Extended thinking consumes additional tokens. It is highly cost-effective for multi-step reasoning, mathematical proofs, or complex coding, but wasteful for simple lookups or classification.

 **2. How They Work Together (The Composition Strategy)**

Model choice and reasoning modes operate as **two distinct, complementary levers** that compose together on every API request:

```text
               ┌─────────────────────────────┐
               │    1. Pick Model Tier       │
               │ (Fable, Opus, Sonnet, Haiku)│
               └──────────────┬──────────────┘
                              │
                              ▼
               ┌─────────────────────────────┐
               │  2. Configure Effort Level  │
               │  (low, medium, high, max)   │
               └──────────────┬──────────────┘
                              │
                              ▼
               ┌─────────────────────────────┐
               │ Optimized API Request Call  │
               └─────────────────────────────┘

```

1. **Lever 1: Model Choice (Baseline Capability)**
* Selects the fundamental intelligence, context limits, and parameter size tier (e.g., **Sonnet** as default, **Opus/Fable** for maximum capability, or **Haiku** for speed/cost).


2. **Lever 2: Effort Setting (Reasoning Intensity)**
* Configures the compute intensity for that specific API call.



**Practical Composition Examples**

* **Haiku + High Effort:** Pair a fast, low-cost model with higher reasoning effort to solve moderately complex tasks at a fraction of the cost of larger models.
* **Sonnet + Low Effort:** Use a mid-tier model with low effort for low-latency, straightforward data extraction or formatting tasks.
* **Opus / Fable + High/Max Effort:** Combine a top-tier model with maximum effort for autonomous coding agents, architecture design, or complex multi-step research.

**Key Takeaway**

Model tier selection sets the *baseline capacity*, while effort setting dictates the *runtime reasoning depth* per execution. Defaults vary across model generations, so always confirm current thinking defaults when building your workflow.

### Prompting Modes

**Key Prompting Modes & Examples**

**1. Zero-Shot**
- Concept: You describe the task directly via instructions without providing any prior examples. You rely entirely on the model's pre-trained knowledge to understand the task and format the response correctly.

- When to Use: Simple, standard tasks, open-ended creative tasks, or when using larger models that don't require structural guidance.

Example:

    Prompt:
    Classify the sentiment of the following customer feedback as Positive, Neutral, or Negative.
    "The package arrived two days late and the box was damaged."
    Response: Negative

**2. One-Shot**
- Concept: You include one worked example showing a sample input paired with its target output before providing the actual input you want processed.

- When to Use: When a plain text instruction alone fails to specify the exact output format, schema, or tone required.

Example:

    Prompt:
    Extract the user name and issue from the support ticket.
    Example:
    Input: "Hi, I'm Sarah and I can't reset my password."
    Output: {"name": "Sarah", "issue": "password_reset"}
    Input: "Hello, my name is Alex and I was charged twice for my subscription."
    Output:
    Response: {"name": "Alex", "issue": "double_charge"}

**3. Multi-Shot (Few-Shot)**

- Concept: You include multiple worked examples (typically 3–5+) showing several variations of input-output pairs to demonstrate patterns, edge cases, and strict formatting rules.
- When to Use: Complex classification tasks, consistent structural parsing, legacy format conversions, or when guiding smaller/haiku models to match the precision of larger models.

Example:

    Prompt:Convert product technical names into customer-friendly categories.
    Example 1:Input: "AC-PWR-CBL-10FT" $\rightarrow$ Category: Accessories, Item: 10ft Power Cable
    Example 2:Input: "MON-27in-4K-OLED" $\rightarrow$ Category: Displays, Item: 27-inch 4K OLED Monitor
    Example 3:Input: "KB-WIRELESS-RGB" $\rightarrow$ Category: Peripherals, Item: Wireless RGB Keyboard
    Input: "HD-EXT-2TB-SSD" $\rightarrow$Response: Category: Storage, Item: 2TB External SSD

**Cost vs. Quality Trade-off:**

- Zero-shot uses fewer tokens (lower cost) but offers less format guarantee.

- Multi-shot uses significantly more input tokens (higher cost), but dramatically reduces output variance and parsing errors.

### Technical Substrate

**Summary: Official SDK vs. Raw REST API**

In Module 1 (Technical Substrate), access to Claude models is divided into direct HTTP REST API calls versus using official Client SDKs:

1. Raw REST API
    - Mechanism: Direct HTTP requests sent via any HTTP client (e.g., curl, requests, fetch) to api.anthropic.com.
    - Requirements: You must manually assemble JSON payloads, format headers (such as x-api-key and anthropic-version), and handle HTTP status codes directly.
    - Best Used For: Custom environments or programming languages where no official Anthropic SDK exists, or when writing zero-dependency microservices.


2. Official SDKs (Python & TypeScript)
    - Mechanism: A thin wrapper provided by Anthropic that interfaces directly with the underlying REST API endpoints.
    - Features Provided Out-of-the-Box:
        - Automated Request Construction: Automatically manages headers, serialization, and typing.
        - Built-in Security & Auth: Handles authentication securely.
        - Error Handling & Retries: Built-in exponential backoff retries for transient errors (e.g., rate limits, network drops).
        - Response Parsing: Converts raw JSON strings directly into typed response objects or stream event iterators.

Key TakeawayBoth methods talk to the exact same backend endpoints and underlying Claude models. The SDK simply eliminates developer boilerplate so you can focus on writing core logic instead of managing network infrastructure.

**Summary: Synchronous, Streaming, and Real-Time Responses**

This section covers the communication models used to interact with Claude depending on latency requirements and user interface needs:

**Synchronous Requests:**

- How it works: You send a request and wait until the model finishes generating the entire response before receiving it back in one complete payload.

- When to use: Ideal for short responses or background/backend processing tasks where no user is actively waiting for real-time visual output.

**Streaming Responses:**

- How it works: Sent as Server-Sent Events (SSE) over the HTTP connection, delivering response tokens incrementally as they are generated. Your application reassembles these chunks into the final message.

- When to use: Crucial when generating long responses or building user-facing interfaces (like chat apps), eliminating the "blank-screen wait" and showing instant progress.

Deep Dive Note: Resilient handling and stream interruption recovery are covered further in Module 2.

**Asynchronous Execution**

Asynchronous execution patterns allow applications to handle high volumes of API requests efficiently by preventing the main application thread from blocking while waiting for LLM responses.

Depending on your latency and workflow requirements, there are two main asynchronous patterns: Non-blocking SDK Async Calls and the Message Batches API.

**1. Non-Blocking SDK Async Calls (Real-Time Concurrency)**
_Best Used For:_ Real-time applications (web servers, user-facing apps, chat agents) where you need to process multiple user requests concurrently without freezing the application.

_How It Works:_ The request is sent to the API immediately over a standard connection. Instead of blocking thread execution, your code uses non-blocking async/await constructs so the CPU can handle other incoming requests while waiting for the network response.

Cost & Latency: Regular API pricing; immediate real-time response.

Python Example (AsyncAnthropic)
Python

    import asyncio
    from anthropic import AsyncAnthropic
    client = AsyncAnthropic()


    async def process_prompt(prompt: str):
        # Non-blocking call to the Messages API
        response = await client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=100,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text


    async def main():
        prompts = [
            "Summarize quantum computing.",
            "Explain async/await in Python.",
            "What is tokenization?",
        ]

        # Process all 3 requests concurrently without blocking
        results = await asyncio.gather(*(process_prompt(p) for p in prompts))

        for prompt, result in zip(prompts, results):
            print(f"Prompt: {prompt}\nResponse: {result}\n---")


    asyncio.run(main())

**2. Message Batches API (Bulk Offline Workloads)**
Best Used For: Offline, bulk background processing where immediate responses are not required (e.g., evaluation benchmark runs, nightly data enrichment, bulk document summarization).

How It Works: You compile a large batch of requests into a single batch submission, receive a batch_id, and poll for completion asynchronously.

Cost & Latency: Jobs complete within up to 24 hours, but offer a 50% discount on per-token pricing compared to standard API calls.

Python Example (client.beta.messages.batches)
Python
    from anthropic import Anthropic

    client = Anthropic()

    # 1. Create a bulk request batch
    batch = client.beta.messages.batches.create(
        requests=[
            {
                "custom_id": "doc_1",
                "params": {
                    "model": "claude-3-5-sonnet-20241022",
                    "max_tokens": 500,
                    "messages": [
                        {
                            "role": "user",
                            "content": "Summarize user document 1...",
                        }
                    ],
                },
            },
            {
                "custom_id": "doc_2",
                "params": {
                    "model": "claude-3-5-sonnet-20241022",
                    "max_tokens": 500,
                    "messages": [
                        {
                            "role": "user",
                            "content": "Summarize user document 2...",
                        }
                    ],
                },
            },
        ]
    )

    print(f"Batch created with ID: {batch.id}")

    # 2. Check batch status periodically
    batch_status = client.beta.messages.batches.retrieve(batch.id)
    print(f"Batch processing status: {batch_status.processing_status}")